# Concurrency & Recovery Demos

Maps Neo4j behavior to Elmasri & Navathe ch. 21–22. Runs against the 10-patient subset loaded by `load_subset.py`.

## Prerequisites

- `docker compose up -d` — `neo4j-demo` container running
- `python load_subset.py` — subset loaded into Neo4j

## Schema

- `(:PATIENT {patient_id, clinician_note, note_embedding})`
- `(:IMAGE {instance_uid, patient_id, image_link, series_description, ...DICOM})`
- `(:PATIENT)-[:HAS_IMAGE]->(:IMAGE)`
- Constraints: `PATIENT.patient_id` UNIQUE, `IMAGE.instance_uid` UNIQUE

## Demo isolation

Demos write only to demo-scoped properties on real PATIENT nodes: `p.demo_note`, `p.demo_last_visit`, `p.demo_visit_count`, `p.demo_run_id`. Ephemeral entities carry the `:DemoNode` label. Real `clinician_note` and `IMAGE.*` are never touched. The setup cell cleans up prior demo state so the notebook is idempotent.

## Log evidence is printed in-cell

After each demo runs, the cell calls `show_log(...)` to dump the relevant slice of `/logs/query.log` or `/logs/debug.log` from inside the container — no terminal-tailing needed. Each demo tags its transactions with `demo_run_id = RUN_ID`, which appears in the query log's metadata field and is used as the grep filter.

## Demo order

| # | Theme       | Demo                                  |
|---|-------------|---------------------------------------|
| 1 | Transaction | Atomic multi-entity rollback          |
| 2 | Transaction | Checkpoint + WAL truncation           |
| 3 | Transaction | Crash recovery via WAL replay         |
| 4 | Concurrency | Non-repeatable read (READ COMMITTED)  |
| 5 | Concurrency | Lost update (naive RMW vs atomic SET) |


In [ ]:
import os, time, uuid, shlex, subprocess
from threading import Event, Thread

import neo4j
from neo4j.exceptions import ConstraintError
from dotenv import load_dotenv

load_dotenv()
URI       = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
USER      = os.getenv("NEO4J_USERNAME", "neo4j")
PWD       = os.getenv("NEO4J_PASSWORD", "password123")
DB        = os.getenv("NEO4J_DATABASE", "neo4j")
CONTAINER = "neo4j-demo"
RUN_ID    = f"nb-{uuid.uuid4().hex[:8]}"


def new_driver():
    d = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD))
    d.verify_connectivity()
    return d


def tx_meta(label):
    return {"demo_run_id": RUN_ID, "tx_label": label}


def show_log(logfile, n=40, grep=None, max_line=240):
    if grep:
        cmd = f"grep -iE {shlex.quote(grep)} /logs/{logfile} 2>/dev/null | tail -n {n}"
        header = f"--- /logs/{logfile} matching {grep!r}, last {n} ---"
    else:
        cmd = f"tail -n {n} /logs/{logfile}"
        header = f"--- /logs/{logfile} last {n} lines ---"
    p = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c", cmd],
        capture_output=True, text=True, timeout=15,
    )
    print(header)
    out = p.stdout or "(no matches)"
    if max_line:
        out = "\n".join((ln if len(ln) <= max_line else ln[:max_line] + " ...[truncated]")
                        for ln in out.splitlines())
    print(out)


def cleanup_demo_state(driver):
    with driver.session(database=DB) as s:
        s.run("MATCH (n:DemoNode) DETACH DELETE n").consume()
        s.run("""
        MATCH (p:PATIENT)
        WHERE p.demo_note        IS NOT NULL
           OR p.demo_last_visit  IS NOT NULL
           OR p.demo_visit_count IS NOT NULL
           OR p.demo_run_id      IS NOT NULL
        REMOVE p.demo_note, p.demo_last_visit,
               p.demo_visit_count, p.demo_run_id
        """).consume()


driver = new_driver()
cleanup_demo_state(driver)

with driver.session(database=DB) as s:
    rows = s.execute_read(lambda tx: [dict(r) for r in tx.run("""
        MATCH (p:PATIENT) WHERE NOT p:DemoNode
        OPTIONAL MATCH (p)-[:HAS_IMAGE]->(i:IMAGE)
        WITH p, head(collect(i.instance_uid)) AS uid
        RETURN p.patient_id AS pid, uid
        ORDER BY pid LIMIT 25
    """)])

if not rows:
    raise RuntimeError("No :PATIENT nodes. Run `python load_subset.py`.")

REAL_PID_A = rows[0]["pid"]
EXISTING_INSTANCE_UID = next((r["uid"] for r in rows if r["uid"]), None)
if EXISTING_INSTANCE_UID is None:
    raise RuntimeError("No :IMAGE nodes found. Run `python load_subset.py`.")

print(f"demo_run_id = {RUN_ID}")
print(f"patient_ids available: {[r['pid'] for r in rows]}")
print(f"REAL_PID_A = {REAL_PID_A}")
print(f"sample instance_uid: {EXISTING_INSTANCE_UID}")


## Demo 1 — Atomic multi-entity transaction with rollback

**Concept:** ACID atomicity. A transaction is all-or-nothing.

**Workflow inside one transaction:**
1. `SET p.demo_last_visit = <new>` on a real PATIENT
2. `CREATE (:Annotation:DemoNode)`
3. Link the annotation to the PATIENT and one of its real IMAGEs
4. **[injected failure]** `CREATE (:IMAGE {instance_uid: <existing>})` — violates the `IMAGE.instance_uid` UNIQUE constraint from `load_subset.py`

**Expected:** step 4 raises `ConstraintError`, rollback fires, the BEFORE and AFTER snapshots are identical. No orphan node, no edge, no timestamp change.

**Log evidence printed below:** `query.log` lines tagged with this run's `demo_run_id` — you should see the staged statements followed by a rolled-back transaction.

**Maps to:** Elmasri & Navathe ch. 21, atomicity.

In [ ]:
with driver.session(database=DB) as s:
    s.execute_write(lambda tx: tx.run(
        "MATCH (p:PATIENT {patient_id: $pid}) "
        "SET p.demo_last_visit = datetime('2025-01-01T00:00:00Z'), p.demo_run_id = $rid",
        pid=REAL_PID_A, rid=RUN_ID,
    ))


def snapshot():
    cy = """
    MATCH (p:PATIENT {patient_id: $pid})
    OPTIONAL MATCH (a:Annotation:DemoNode {demo_run_id: $rid})
    OPTIONAL MATCH (:Annotation:DemoNode {demo_run_id: $rid})-[r:ANNOTATES]->
                   (:PATIENT {patient_id: $pid})
    RETURN toString(p.demo_last_visit) AS last_visit,
           count(DISTINCT a) AS annotations,
           count(DISTINCT r) AS edges
    """
    with driver.session(database=DB) as s:
        return s.execute_read(lambda tx: dict(
            tx.run(cy, pid=REAL_PID_A, rid=RUN_ID).single()
        ))


before = snapshot()
print(f"BEFORE: {before}")

annotation_id = f"{RUN_ID}-rb-{int(time.time())}"
failure = None
with driver.session(database=DB) as s:
    tx = s.begin_transaction(metadata=tx_meta("rollback-demo"))
    try:
        tx.run(
            "MATCH (p:PATIENT {patient_id: $pid}) "
            "SET p.demo_last_visit = datetime('2026-05-14T10:30:00Z')",
            pid=REAL_PID_A,
        )
        tx.run(
            """
            CREATE (a:Annotation:DemoNode {
                annotation_id: $aid,
                text:          'Suspicious finding at L4-L5',
                demo_run_id:   $rid,
                created_at:    datetime()
            })
            """,
            aid=annotation_id, rid=RUN_ID,
        )
        tx.run(
            """
            MATCH (a:Annotation {annotation_id: $aid}),
                  (p:PATIENT    {patient_id:   $pid})-[:HAS_IMAGE]->(i:IMAGE)
            WITH a, p, i LIMIT 1
            MERGE (a)-[:ANNOTATES]->(p)
            MERGE (a)-[:ABOUT_IMAGE]->(i)
            """,
            aid=annotation_id, pid=REAL_PID_A,
        )
        print("steps 1/2/3 staged")

        tx.run(
            "CREATE (:IMAGE {instance_uid: $uid, patient_id: $pid})",
            uid=EXISTING_INSTANCE_UID, pid=REAL_PID_A,
        )
        tx.commit()
    except ConstraintError as e:
        failure = e
        tx.rollback()
        print(f"ConstraintError -> rollback: {e.message}")
    finally:
        tx.close()

after = snapshot()
print(f"AFTER : {after}")

assert failure is not None
assert before == after, "Atomicity violated"
print("\nACID atomicity verified: failed transaction is a no-op.\n")

show_log("query.log", n=20, grep=RUN_ID)


## Demo 2 — Checkpoint and WAL truncation

**Concept:** every committed change is appended to the write-ahead log under `/data/transactions/neo4j/`. The WAL grows monotonically. Periodically, a CHECKPOINT (1) flushes dirty pages to the main store, (2) writes a checkpoint marker into the WAL, (3) lets older WAL segments be pruned.

**Why it matters:** crash recovery only needs to replay from the *last checkpoint* forward, not from db creation. That's the optimization that bounds recovery time — making the next demo's restart fast.

**Action:** snapshot the tx-log directory, run 200 small writes, snapshot again, trigger `CALL db.checkpoint()`, snapshot once more.

**Log evidence printed below:** `debug.log` lines about `Checkpoint started/completed` and log pruning.

**Maps to:** Elmasri & Navathe ch. 22, checkpointing.

In [ ]:
def list_tx_files():
    p = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c",
         f"ls -la /data/transactions/{DB}/ 2>/dev/null"],
        capture_output=True, text=True,
    )
    return p.stdout or p.stderr


print("BEFORE writes:")
print(list_tx_files())

N = 200
with driver.session(database=DB) as s:
    for i in range(N):
        s.execute_write(lambda tx: tx.run(
            "MATCH (p:PATIENT {patient_id: $pid}) "
            "SET p.demo_note = $v, p.demo_run_id = $rid",
            pid=REAL_PID_A, v=f"tick-{i}", rid=RUN_ID,
        ))
print(f"Ran {N} small write transactions.\n")

print("AFTER writes (before explicit checkpoint):")
print(list_tx_files())

try:
    with driver.session(database=DB) as s:
        print("CALL db.checkpoint() ->", s.run("CALL db.checkpoint()").data())
except Exception as e:
    print(f"db.checkpoint() unavailable: {e}")
    print("  (waiting 8s for an automatic checkpoint)")
    time.sleep(8)

print("\nAFTER checkpoint:")
print(list_tx_files())

print()
show_log("debug.log", n=20, grep="checkpoint|prun")


## Demo 3 — Crash recovery via WAL replay (durability)

**Concept:** durability. Once a transaction commits, its effects survive any subsequent failure — including a kernel-level kill of the database process.

**Steps:**
1. Commit a `:CrashMarker` node.
2. `docker kill -s SIGKILL neo4j-demo` — no graceful shutdown, no flush.
3. `docker start` and poll Bolt until it's back.
4. Print `debug.log` recovery lines (printed in-cell, see below).
5. Query for the marker — it must still be there.

**Local-only:** Aura doesn't let you SIGKILL the server. This is the strongest case for the Docker setup.

In [ ]:
marker_id = f"{RUN_ID}-CRASH-{int(time.time())}"

with driver.session(database=DB) as s:
    s.execute_write(lambda tx: tx.run(
        "CREATE (m:CrashMarker:DemoNode {marker_id: $mid, "
        "demo_run_id: $rid, written_at: datetime()})",
        mid=marker_id, rid=RUN_ID,
    ))
print(f"Committed CrashMarker {marker_id!r}")

driver.close()
print(f"docker kill -s SIGKILL {CONTAINER}")
subprocess.run(["docker", "kill", "-s", "SIGKILL", CONTAINER],
               check=True, capture_output=True, text=True)

print(f"docker start {CONTAINER}")
subprocess.run(["docker", "start", CONTAINER],
               check=True, capture_output=True, text=True)

driver = None
deadline = time.time() + 90
while time.time() < deadline:
    try:
        driver = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD))
        driver.verify_connectivity()
        break
    except Exception:
        if driver: driver.close()
        driver = None
        time.sleep(1)
if driver is None:
    raise RuntimeError("Neo4j didn't recover within 90s")
print("Bolt is back.\n")

show_log("debug.log", n=40, grep="recover|replay")

with driver.session(database=DB) as s:
    rec = s.execute_read(lambda tx: tx.run(
        "MATCH (m:CrashMarker {marker_id: $mid}) "
        "RETURN m.marker_id AS id, toString(m.written_at) AS t",
        mid=marker_id,
    ).single())
    pcount = s.execute_read(lambda tx: tx.run(
        "MATCH (p:PATIENT) WHERE NOT p:DemoNode RETURN count(p) AS n"
    ).single()["n"])

print(f"\nMarker recovered: {rec}")
print(f"Real :PATIENT count after recovery: {pcount}")
assert rec is not None, "Durability violated"


## Demo 4 — Non-repeatable read under READ COMMITTED

**Concept:** Neo4j's default isolation level is READ COMMITTED. That prevents *dirty* reads (you never see uncommitted writes) but does **not** prevent *non-repeatable* reads — two reads of the same property in the same transaction can return different values if another transaction commits between them.

**Timeline:**

```
READER tx:  read #1 ---------------------------- read #2 -- commit
WRITER tx:           SET ... -- commit
```

**Log evidence printed below:** `query.log` lines tagged with this run's `demo_run_id` — you should see the READER and WRITER transactions interleave (the READER reads twice, the WRITER commits between them).

**Maps to:** Elmasri & Navathe ch. 21, isolation levels — the anomaly READ COMMITTED *doesn't* prevent.

In [ ]:
with driver.session(database=DB) as s:
    s.execute_write(lambda tx: tx.run(
        "MATCH (p:PATIENT {patient_id: $pid}) "
        "SET p.demo_note = 'initial-nrr', p.demo_run_id = $rid",
        pid=REAL_PID_A, rid=RUN_ID,
    ))

first_done   = Event()
writer_done  = Event()
results: dict = {}


def reader():
    with driver.session(database=DB) as s:
        tx = s.begin_transaction(metadata=tx_meta("READER"))
        try:
            r1 = tx.run("MATCH (p:PATIENT {patient_id: $pid}) RETURN p.demo_note AS n",
                        pid=REAL_PID_A).single()["n"]
            results["r1"] = r1
            print(f"[READER] read #1 = {r1!r}")
            first_done.set()

            writer_done.wait(timeout=10)

            r2 = tx.run("MATCH (p:PATIENT {patient_id: $pid}) RETURN p.demo_note AS n",
                        pid=REAL_PID_A).single()["n"]
            results["r2"] = r2
            print(f"[READER] read #2 = {r2!r}   (same tx, after WRITER committed)")
            tx.commit()
        except Exception:
            tx.rollback(); raise


def writer():
    first_done.wait()
    with driver.session(database=DB) as s:
        s.execute_write(lambda tx: tx.run(
            "MATCH (p:PATIENT {patient_id: $pid}) "
            "SET p.demo_note = 'mutated-by-writer'",
            pid=REAL_PID_A,
        ))
    print("[WRITER] committed")
    writer_done.set()


t_r = Thread(target=reader, daemon=True)
t_w = Thread(target=writer, daemon=True)
t_r.start(); t_w.start()
t_r.join();  t_w.join()

print(f"\nr1={results.get('r1')!r}   r2={results.get('r2')!r}")
if results.get("r1") != results.get("r2"):
    print("Non-repeatable read confirmed: same property, same tx, different values.\n")
else:
    print("Reads matched - threads serialised. Re-run.\n")

show_log("query.log", n=20, grep=RUN_ID)


## Demo 5 — Lost update: naive RMW vs atomic Cypher

**Concept:** when application code does its own read-modify-write across multiple statements, the read does *not* take a lock — the lock is only taken on the write. Two threads can read the same value and both write back `value+1`. One increment is silently lost.

**Two rounds:**

| | Pattern | Expected final | What actually happens |
|---|---|---|---|
| Naive | `v = SELECT; SET = v+1` (two statements) | 200 | < 200 (lost updates) |
| Atomic | `SET p.counter = p.counter + 1` (one statement) | 200 | exactly 200 |

**Why retries don't save you:** lost updates raise no error, so `execute_write`'s retry logic cannot trigger. Only correct Cypher can.

**Log evidence printed below:** count of `query.log` lines tagged with this run's `demo_run_id`, plus the tail of the log so you can see the SET statements going through.

**Maps to:** Elmasri & Navathe ch. 21, the canonical *lost-update* anomaly.

The final line cleans up all demo state so the notebook leaves no residue on the real subset.

In [ ]:
N = 100


def naive(pid, n):
    with driver.session(database=DB) as s:
        for _ in range(n):
            def work(tx):
                v = tx.run(
                    "MATCH (p:PATIENT {patient_id: $pid}) "
                    "RETURN coalesce(p.demo_visit_count, 0) AS v",
                    pid=pid,
                ).single()["v"]
                time.sleep(0.003)
                tx.run(
                    "MATCH (p:PATIENT {patient_id: $pid}) "
                    "SET p.demo_visit_count = $v",
                    pid=pid, v=v + 1,
                )
            s.execute_write(work)


def atomic(pid, n):
    with driver.session(database=DB) as s:
        for _ in range(n):
            s.execute_write(lambda tx: tx.run(
                "MATCH (p:PATIENT {patient_id: $pid}) "
                "SET p.demo_visit_count = coalesce(p.demo_visit_count, 0) + 1",
                pid=pid,
            ))


def reset():
    with driver.session(database=DB) as s:
        s.execute_write(lambda tx: tx.run(
            "MATCH (p:PATIENT {patient_id: $pid}) "
            "SET p.demo_visit_count = 0, p.demo_run_id = $rid",
            pid=REAL_PID_A, rid=RUN_ID,
        ))


def final_value():
    with driver.session(database=DB) as s:
        return s.execute_read(lambda tx: tx.run(
            "MATCH (p:PATIENT {patient_id: $pid}) RETURN p.demo_visit_count AS v",
            pid=REAL_PID_A,
        ).single()["v"])


reset()
print(f"NAIVE RMW, 2 threads x {N} (expected {2*N}):")
t1 = Thread(target=naive, args=(REAL_PID_A, N), daemon=True)
t2 = Thread(target=naive, args=(REAL_PID_A, N), daemon=True)
t1.start(); t2.start(); t1.join(); t2.join()
naive_v = final_value()
print(f"  -> {naive_v}   (lost {2*N - naive_v})\n")

reset()
print(f"ATOMIC SET expression, 2 threads x {N} (expected {2*N}):")
t1 = Thread(target=atomic, args=(REAL_PID_A, N), daemon=True)
t2 = Thread(target=atomic, args=(REAL_PID_A, N), daemon=True)
t1.start(); t2.start(); t1.join(); t2.join()
atomic_v = final_value()
print(f"  -> {atomic_v}\n")

cnt = subprocess.run(
    ["docker", "exec", CONTAINER, "sh", "-c",
     f"grep -c {shlex.quote(RUN_ID)} /logs/query.log || true"],
    capture_output=True, text=True,
)
print(f"query.log lines tagged with this run: {cnt.stdout.strip()}")
show_log("query.log", n=8, grep=RUN_ID)

cleanup_demo_state(driver)
print("\nDemo state cleaned.")
